# English audio render — Don's voice (F5) + post-process, commit-per-article

For every audio-eligible library + blog article, this: **(1) renders** the English narration in
your cloned voice (F5-TTS), **(2) post-processes** it (EQ / de-ess / compress / −16 LUFS loudnorm /
ambient bed — `scripts/audio-post-process.mjs`), and **(3) commits + pushes the finished MP3
immediately**. If Colab disconnects you lose at most the one article in flight; to resume, re-run
cells **1–6** then the render cell — it skips everything already pushed.

Run top to bottom (Shift+Enter). First: `Runtime → Change runtime type → T4 GPU → Save`. Your voice
reference is already in the repo, so there's nothing to upload.

## 1. Confirm GPU
Output should mention `Tesla T4` (or similar).

In [ ]:
!nvidia-smi

## 2. Install Node 20 + F5-TTS + ffmpeg (~2 min)
`ffmpeg` is used by **both** the render and the post-process; F5-TTS does the voice cloning.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs ffmpeg -qq > /dev/null
!node --version && ffmpeg -version | head -1
!pip install -q f5-tts jieba

## 3. Add your GitHub PAT — the obvious place is the 🔑 **Secrets** panel
On the left sidebar of Colab, click the **key (🔑) icon → "+ Add new secret"**. Name it exactly
`GITHUB_PAT`, paste your token (needs **repo** scope) as the value, and switch **Notebook access**
ON. The next cell reads it from there — it is never written into the notebook.

(No Secrets panel? The cell falls back to a hidden paste-prompt that appears *below the cell when
you run it*.)

In [ ]:
import os, subprocess
BRANCH = 'claude/compassionate-dirac-rdkw22'
REPO   = 'github.com/Donwonmagic/potentially-profitable.git'

token = None
try:
    from google.colab import userdata
    token = (userdata.get('GITHUB_PAT') or '').strip()
    if token: print('using token from the Secrets panel (GITHUB_PAT)')
except Exception:
    token = None
if not token:
    import getpass
    token = getpass.getpass('Paste GitHub token (repo scope) here, then Enter: ').strip()
assert token, 'no token — add GITHUB_PAT to the Secrets panel (step 3) and re-run'

url = f'https://x-access-token:{token}@{REPO}'
!rm -rf /content/potentially-profitable
subprocess.run(['git','clone','-b',BRANCH,'--depth','1',url,'/content/potentially-profitable'], check=True)
%cd /content/potentially-profitable
!git config user.name 'Don Goldstein'
!git config user.email 'dongoldstein.accts@gmail.com'
# F5 hardcodes torch.xpu (Intel GPU); Colab's CUDA build lacks it. Make the check safe.
import glob
for f in glob.glob('/usr/local/lib/python3*/dist-packages/f5_tts/**/*.py', recursive=True):
    s = open(f, encoding='utf-8').read()
    if 'torch.xpu.is_available()' in s:
        open(f,'w',encoding='utf-8').write(s.replace('torch.xpu.is_available()', '(hasattr(torch,"xpu") and torch.xpu.is_available())'))
print('cloned', BRANCH, '+ F5 patched')

## 4. Verify your voice reference is in the repo

In [ ]:
import os
assert os.path.isfile('scripts/voice-refs/don-reference.m4a'), 'voice clip missing'
assert os.path.isfile('scripts/voice-refs/don-reference.txt'), 'transcript missing'
print('voice reference present:', os.path.getsize('scripts/voice-refs/don-reference.m4a'), 'bytes')

## 5. Build the article list + resume marker
Every English library + blog article with a listen button. Edit `targets` to render a subset.
`CUTOFF` = anything already F5-rendered on/after this is treated as done (makes the render resumable).

In [ ]:
import glob, os
def has_listen(p):
    try: return 'id="listen-btn"' in open(p, encoding='utf-8').read()
    except Exception: return False
targets = []
for base in ('library','blog'):
    for d in sorted(glob.glob(base + '/*/')):
        idx = os.path.join(d, 'index.html')
        if os.path.isfile(idx) and has_listen(idx):
            targets.append(d.rstrip('/'))
CUTOFF = '2026-06-22T00:00:00'
print(len(targets), 'audio-eligible English articles')
for t in targets: print('  ', t)

## 6. Render → post-process → commit, per article (the long one, hours)
Per article: skip if already F5-rendered on/after `CUTOFF`; else render the English track in your
voice, run the post-process chain on `audio.mp3`, then commit + push **just** `audio.mp3` +
`audio.json` (never the `.raw.mp3` backup). A failure on one article cleans up its half-written
output and moves on, so the next run re-does only that one. The F5 base model (~1.5 GB) downloads
on the first article.

**Resume after a disconnect:** re-run cells 1–6, then this cell.

In [ ]:
import os, json, subprocess, time

def is_done(d):
    j = os.path.join(d, 'audio.json')
    if not os.path.isfile(j) or not os.path.isfile(os.path.join(d, 'audio.mp3')): return False
    try: m = json.load(open(j, encoding='utf-8'))
    except Exception: return False
    return str(m.get('engine','')) == 'f5' and str(m.get('generatedAt','')) >= CUTOFF

def run(cmd):
    return subprocess.run(cmd).returncode

def commit_push(d):
    run(['git','add', f'{d}/audio.mp3', f'{d}/audio.json'])
    if subprocess.run(['git','diff','--cached','--quiet']).returncode == 0: return True
    if run(['git','commit','-m', f'audio: {d}']) != 0: return False
    for a in range(3):
        if run(['git','push','origin',BRANCH]) == 0: return True
        time.sleep(2*(a+1))
    return False

def cleanup(d):
    for f in ('audio.mp3','audio.json','audio.raw.mp3'):
        p = os.path.join(d, f)
        if os.path.isfile(p): os.remove(p)

done = skipped = failed = 0; fails = []
for i, d in enumerate(targets, 1):
    if is_done(d):
        skipped += 1; print(f'[{i}/{len(targets)}] skip (already f5): {d}'); continue
    print(f'[{i}/{len(targets)}] render: {d}', flush=True)
    if run(['node','scripts/render-post-audio.mjs', d, '--engine','f5','--languages','en','--force-retranslate']) != 0:
        failed += 1; fails.append(d); cleanup(d); print('  !! render failed — cleaned up, continuing'); continue
    print(f'    post-process: {d}/audio.mp3', flush=True)
    if run(['node','scripts/audio-post-process.mjs', f'{d}/audio.mp3']) != 0:
        failed += 1; fails.append(d); cleanup(d); print('  !! post-process failed — cleaned up, continuing'); continue
    if not commit_push(d):
        failed += 1; fails.append(d); cleanup(d); print('  !! commit/push failed — cleaned up, continuing'); continue
    done += 1; print(f'    committed + pushed: {d}')
print(f'\n=== rendered {done}, skipped {skipped}, failed {failed} ===')
if fails: print('failed (will retry on next run):', fails)

## 7. Done
Every article you see above is committed and pushed to `claude/compassionate-dirac-rdkw22` as
`audio: <slug>` — post-processed MP3 + manifest. Nothing to download or commit by hand; the live
player picks up the new audio on the next deploy.

Spanish later: same loop with `--languages es` (+ the F5-Spanish checkpoint — see
`scripts/voice-refs/README.md`). Post-process flags worth knowing: `--no-bed` (drop the ambient
bed) and `--no-intro` (skip the branded opener) if the default chain is too much for the cloned voice.